# Week 37, Day 2: Calcium imaging: from fluorescence to responses

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w037/d2.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Correcting the trace

Write delta_f(F_trace): estimate the baseline F0 as a running 10th percentile over 60 s windows (percentile_filter), and return (F_trace - F0) / F0. Apply it to the neuropil-corrected trace F - 0.7 Fneu, and to the raw F for comparison.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import percentile_filter, gaussian_filter1d
r = np.random.default_rng(2020)
fs = 30.0; T = int(600 * fs); t = np.arange(T) / fs
g = np.exp(-1 / (fs * 0.7))                       # decay per frame, tau = 0.7 s
spikes = r.poisson(0.5 / fs, T)
calcium = np.zeros(T)
for i in range(1, T):
    calcium[i] = g * calcium[i - 1] + spikes[i]
drift = 1 + 0.2 * np.sin(2 * np.pi * t / 300)
neuropil = 0.6 + 0.25 * np.sin(2 * np.pi * t / 41) + 0.15 * np.sin(2 * np.pi * t / 7.3) + 0.05 * r.normal(size=T)
F = 100 * drift * (1 + 0.5 * calcium) + 70 * neuropil + 3 * r.normal(size=T)
Fneu = 100 * neuropil + 3 * r.normal(size=T)

def delta_f(F_trace, window_s=60):
    F0 = ...
    return ...

dff = delta_f(F - 0.7 * Fneu)
dff_raw = delta_f(F)
print(f"correlation with calcium: corrected {np.corrcoef(dff, calcium)[0, 1]:.3f}, raw {np.corrcoef(dff_raw, calcium)[0, 1]:.3f}")
sl = slice(0, int(60 * fs))
plt.plot(t[sl], dff_raw[sl], lw=0.7, label="raw ΔF/F"); plt.plot(t[sl], dff[sl], lw=0.7, label="corrected ΔF/F")
plt.plot(t[sl][spikes[sl] > 0], np.full((spikes[sl] > 0).sum(), -0.3), "|k", label="spikes"); plt.xlabel("s"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_dff():
    "ΔF/F after neuropil correction correlates with the true calcium above 0.95"
    assert np.corrcoef(dff, calcium)[0, 1] > 0.95

def test_better():
    "neuropil correction improves the correlation by more than 0.1"
    assert np.corrcoef(dff, calcium)[0, 1] > np.corrcoef(dff_raw, calcium)[0, 1] + 0.1

test_dff()
test_better()
print("All checks passed")

### Exercise: Simple deconvolution

Compute ev, the non-negative part of dff[t] - g * dff[t - 1] for t from 1 to the end. Compare its smoothed correlation with the true spikes against that of ΔF/F.

In [ ]:
ev = ...
sm = lambda x: gaussian_filter1d(np.asarray(x, float), 3)
print(f"correlation with spikes (smoothed 100 ms): events {np.corrcoef(sm(ev), sm(spikes[1:]))[0, 1]:.2f}, ΔF/F {np.corrcoef(sm(dff[1:]), sm(spikes[1:]))[0, 1]:.2f}")
sl = slice(0, int(20 * fs))
plt.plot(t[1:][sl], ev[sl], label="events"); plt.plot(t[1:][sl], spikes[1:][sl] * 0.5, "|k", ms=12, label="spikes"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_events():
    "smoothed events correlate with smoothed spikes above 0.85"
    assert np.corrcoef(gaussian_filter1d(ev, 3), gaussian_filter1d(spikes[1:].astype(float), 3))[0, 1] > 0.85

def test_vs_dff():
    "events track spike timing better than ΔF/F itself does"
    a = np.corrcoef(gaussian_filter1d(ev, 3), gaussian_filter1d(spikes[1:].astype(float), 3))[0, 1]
    b = np.corrcoef(gaussian_filter1d(dff[1:], 3), gaussian_filter1d(spikes[1:].astype(float), 3))[0, 1]
    assert a > b + 0.2

test_events()
test_vs_dff()
print("All checks passed")

### Exercise: Which cells respond?

resp[i, d, k] holds the mean ΔF/F of cell i on the k-th trial of direction d, and base[i, j] the mean ΔF/F in the grey period before trial j. For each cell, find the preferred direction (largest mean response) and call the cell responsive if a one-sided Welch t-test of its preferred-direction responses against all its grey periods gives p below 0.001.

In [ ]:
from scipy.stats import ttest_ind
dirs = np.arange(0, 360, 45); nrep, ncell = 15, 40
order = r.permutation(np.repeat(np.arange(8), nrep))
fs2 = 30.0; on, off = int(2 * fs2), int(1 * fs2)
Tt = len(order) * (on + off)
pref_true = r.uniform(0, 360, ncell); amp = np.r_[r.uniform(1, 5, 24), np.zeros(16)]
stim = np.full(Tt, -1)
for j, d in enumerate(order):
    stim[j * (on + off) + off: (j + 1) * (on + off)] = d
traces = np.zeros((ncell, Tt))
gg = np.exp(-1 / (fs2 * 0.7))
for i in range(ncell):
    ang = np.deg2rad(dirs[np.maximum(stim, 0)] - pref_true[i])
    rate = 0.3 + np.where(stim >= 0, amp[i] * np.exp(2 * (np.cos(ang) - 1)) + 0.3 * amp[i] * np.exp(2 * (np.cos(ang + np.pi) - 1)), 0)
    s = r.poisson(rate / fs2)
    c = np.zeros(Tt)
    for k in range(1, Tt):
        c[k] = gg * c[k - 1] + s[k]
    traces[i] = 0.25 * c + 0.08 * r.normal(size=Tt)
resp = np.zeros((ncell, 8, nrep)); base = np.zeros((ncell, len(order))); seen = np.zeros(8, int)
for j, d in enumerate(order):
    a = j * (on + off)
    base[:, j] = traces[:, a: a + off].mean(1)
    resp[:, d, seen[d]] = traces[:, a + off: a + off + on].mean(1); seen[d] += 1

mean_resp = resp.mean(2)
pref_idx = mean_resp.argmax(1)
pref_est = dirs[pref_idx]
responsive = np.zeros(ncell, bool)
for i in range(ncell):
    responsive[i] = ...
null = mean_resp[np.arange(ncell), (pref_idx + 4) % 8]
dsi = (mean_resp.max(1) - null) / (mean_resp.max(1) + np.abs(null))
print(f"responsive: {responsive[:24].sum()} of 24 tuned cells, {responsive[24:].sum()} of 16 untuned cells")
print("median DSI of responsive cells:", round(float(np.median(dsi[responsive])), 2))

In [ ]:
import numpy as np
def test_found():
    "at least 20 of the 24 tuned cells are called responsive, and at most 2 of the 16 untuned cells"
    assert responsive[:24].sum() >= 20 and responsive[24:].sum() <= 2

def test_pref():
    "for responsive tuned cells, the preferred direction is within 45 degrees of the true one in at least 80% of cases"
    ok = [abs(((pref_est[i] - pref_true[i] + 180) % 360) - 180) <= 45 for i in range(24) if responsive[i]]
    assert np.mean(ok) >= 0.8

test_found()
test_pref()
print("All checks passed")